# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Driver Standings Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

driver_standings_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("position_text", StringType(), True),
    StructField("points", FloatType(), True),
    StructField("wins", IntegerType(), True),
    StructField("driver_id", StringType(), False),
    StructField("permanent_number", IntegerType(), True),
    StructField("code", StringType(), True),
    StructField("given_name", StringType(), True),
    StructField("family_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("constructor_ids", StringType(), True),
    StructField("constructor_names", StringType(), True),
])

driver_standings_input_path = f"{processed_folder_path}/driver_standings/csv/driver_standings.csv"

driver_standings_df = spark.read \
    .option("header", True) \
    .schema(driver_standings_schema) \
    .csv(driver_standings_input_path)

driver_standings_dropped_df = driver_standings_df.drop("permanent_number", "code", "given_name", "family_name", "date_of_birth", "nationality", "constructor_names")

driver_standings_renamed_df = driver_standings_dropped_df.withColumnRenamed("constructor_ids", "constructor_id")

# 3) Transform Driver Standings Data:

The steps included:

- Fill Null cells with "None".
- Fill Null Position and Position Text (that shows "-") with "0".
- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit, when, col

driver_standings_with_audit_df = driver_standings_renamed_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

driver_standings_date_df = add_ingestion_date(driver_standings_with_audit_df)

driver_standings_position_filled_df = driver_standings_date_df \
    .fillna(0, subset=["position"]) \
    .withColumn(
        "position_text",
        when(col("position_text") == "-", "0").otherwise(col("position_text"))
    )

driver_standings_fill_df = driver_standings_position_filled_df.fillna("None")

driver_standings_final_df = add_surrogate_key(
    driver_standings_fill_df,
    key_column_name="driver_standing_sk",
    hash_columns=["season", "round", "position", "position_text", "points", "wins", "driver_id", 
                  "constructor_id"],
)

print("Final columns going into the write:", driver_standings_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
driver_standings_output_path = f"{processed_folder_path}/driver_standings/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=driver_standings_final_df,
    db_name="f1_processed",
    table_name="driver_standings",
    output_path=driver_standings_output_path,
    merge_key_columns=["season", "driver_id"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(driver_standings_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/driver_standings/delta",
    presentation_directory=f"{presentation_folder_path}/fact_driver_standings/delta",
    db_name="f1_presentation",
    table_name="fact_driver_standings",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_driver_standings/delta"))

# 5) Save backup Driver Standings in CSV format:

In [0]:
import io
import csv

driver_standings_backup_path = f"{presentation_folder_path}/fact_driver_standings/csv/fact_driver_standings.csv"

backup_rows = [row.asDict() for row in driver_standings_final_df.collect()]
backup_fieldnames = driver_standings_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(driver_standings_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {driver_standings_backup_path}")